[Step 11 - Stateful chatbot app]

> **MLCourse - Agentic AI - Memory and State**

> Stage in the capstone: the capstone chatbot remembers prior turns PER SESSION thanks to this.

Time to assemble the parts into an APPLICATION: a persona prompt, a guarded
model, a history-aware wrapper, and an interactive loop - the exact shape the
Step 12 capstone reuses with RAG bolted on.

### What you will learn

1. Compose the chatbot: system persona + MessagesPlaceholder("history") + human input.
2. Wrap it with RunnableWithMessageHistory using a session-histories dict factory.
3. Run an input() loop that is QA-SAFE: MAX_TURNS caps it, 'quit' exits, and a
   closed stdin (EOFError) ends it cleanly instead of hanging automation.
4. Prove per-session separation with two scripted sessions.
5. Export every transcript to DATA / "transcripts.json" for inspection.

### Sections

1. Setup
2. The librarian chain + memory wrapper
3. The capped interactive loop
4. Two scripted sessions - separation proof
5. Transcript export
6. Summary

In [1]:
# --- Section 1: setup ----------------------------------------------------------
from pathlib import Path          # cross-platform paths
import os                         # environment access
import json                       # transcripts export at the end
import re                         # offline stub helpers

def _find_track(start_dir):
    """Climb parent folders until we find (or reach) the dir named 03_agentic_ai."""
    here = Path(start_dir).resolve()
    for candidate in (here, *here.parents):
        if candidate.name == "03_agentic_ai":
            return candidate
        if (candidate / "03_agentic_ai").is_dir():
            return candidate / "03_agentic_ai"
    raise FileNotFoundError("Could not locate the 03_agentic_ai track near %s" % here)

TRACK = _find_track(Path.cwd())
DATA = TRACK / "data"
DATA.mkdir(parents=True, exist_ok=True)

from dotenv import load_dotenv    # keys are optional here; guards handle absence
load_dotenv(TRACK / ".env", override=False)
load_dotenv(override=False)

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("track:", TRACK)
print("data :", DATA)

track: D:\projects\python\MLCourse\03_agentic_ai
data : D:\projects\python\MLCourse\03_agentic_ai\data


In [2]:
# --- Section 2: the librarian chain + memory wrapper ---------------------------
# Persona gives the bot CHARACTER and SCOPE ("librarian of Alice"); the history
# placeholder is where the wrapper will inject this session's past turns.
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

LIBRARIAN_PERSONA = (
    "You are the helpful librarian of Alice: you know Wonderland deeply, recommend "
    "chapters, keep answers short (2-3 sentences), and never invent plot events."
)
LIBRARIAN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", LIBRARIAN_PERSONA),
    MessagesPlaceholder(variable_name="history"),   # wrapper fills this per session
    ("human", "{input}"),
])

from langchain_ollama import ChatOllama
base_llm = ChatOllama(model="llama3.2", temperature=0)   # a little warmth is fine here

LLM_LIVE = False
try:
    base_llm.invoke("Reply with the single word: pong")   # one cheap probe up front
    LLM_LIVE = True
except Exception as exc:
    print("[demo skipped] install/start Ollama and run: ollama pull llama3.2")
    print("   detail: %s: %s" % (type(exc).__name__, exc))

def fake_librarian(prompt_value):
    """OFFLINE STUB so the PIPELINE mechanics run anywhere; swap llm back for real answers.

    Mimics the persona deterministically: greets by remembered name when asked,
    recommends chapter one for reading advice, otherwise echoes politely.
    """
    try:
        msgs = prompt_value.to_messages()
    except Exception:
        msgs = []
    last_human = ""
    for m in reversed(msgs):
        if getattr(m, "type", "") == "human":
            last_human = m.content
            break
    low = last_human.lower()
    if "name" in low:
        found = re.findall(r"name is ([A-Za-z]+)",
                           " ".join(getattr(m, "content", "") for m in msgs),
                           flags=re.IGNORECASE)
        if found:
            return "Welcome back to the Alice collection, %s." % found[-1]
        return "I am afraid you have not told me your name yet."
    if "recommend" in low or "read" in low or "start" in low:
        return "Start at Chapter 1, 'Down the Rabbit-Hole' - everything spirals from there."
    return "[offline stub] Noted: \"%s\" (swap llm back for real answers)" % last_human[:80]

if LLM_LIVE:
    llm = base_llm
else:
    from langchain_core.runnables import RunnableLambda
    llm = RunnableLambda(fake_librarian)
    print(">> running with the OFFLINE STUB model for this session")

chatbot_core = LIBRARIAN_PROMPT | llm | StrOutputParser()

from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

SESSION_HISTORIES = {}                    # {'session-id': InMemoryChatMessageHistory}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """Per-session factory: THE pattern that prevents cross-session leakage."""
    if session_id not in SESSION_HISTORIES:
        SESSION_HISTORIES[session_id] = InMemoryChatMessageHistory()
    return SESSION_HISTORIES[session_id]

chatbot = RunnableWithMessageHistory(
    chatbot_core,
    get_session_history=get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

def guarded_invoke(chain, payload, config=None):
    """Invoke with the standard skip message on any provider failure."""
    try:
        return chain.invoke(payload, config=config)
    except Exception as exc:
        print("[demo skipped] install/start Ollama and run: ollama pull llama3.2")
        print("   detail: %s: %s" % (type(exc).__name__, exc))
        return None

print("librarian chatbot assembled:",
      "persona | history | human -> model -> parser, wrapped per-session")

[demo skipped] install/start Ollama and run: ollama pull llama3.2
   detail: ConnectError: [WinError 10061] No connection could be made because the target machine actively refused it
>> running with the OFFLINE STUB model for this session
librarian chatbot assembled: persona | history | human -> model -> parser, wrapped per-session


D:\projects\python\MLCourse\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [3]:
# --- Section 3: the capped interactive loop ------------------------------------
# WHY the guards: input() blocks forever under automation, which would hang CI/QA
# runs of this notebook. Three safeties make execution deterministic:
#   MAX_TURNS hard cap, explicit exit commands, EOFError when stdin is closed.
INTERACTIVE = False                       # set True for a real terminal chat
SCRIPTED_INPUTS = ['What made Alice grow so tall?',
                   'And how did she shrink again?',
                   'quit']                # deterministic QA turns
MAX_TURNS = 3                             # learners: raise me for a real chat
EXIT_COMMANDS = {"quit", "exit", "q"}

class _StdinClosed(Exception):
    """Raised by our input shim when no interactive terminal exists."""

def _input_or_script(prompt):
    # Under nbconvert there is no stdin: StdinNotImplementedError fires.
    # We translate that into our own sentinel so the loop stays testable.
    try:
        return input(prompt).strip()
    except Exception as exc:              # StdinNotImplementedError lives here
        if type(exc).__name__ == "StdinNotImplementedError":
            raise _StdinClosed
        raise

def chat_loop(session_id: str, max_turns=MAX_TURNS):
    """Chat loop; returns THIS session's user/bot transcript list."""
    cfg = {"configurable": {"session_id": session_id}}
    transcript = []
    print("=" * 64)
    print("chat session '%s' - cap %d turns" % (session_id, max_turns))
    for turn_no in range(1, max_turns + 1):
        try:
            if INTERACTIVE:
                user_text = _input_or_script("you > ")
            else:
                user_text = (SCRIPTED_INPUTS.pop(0)
                             if SCRIPTED_INPUTS else "quit")
                print("you >", user_text)
        except (_StdinClosed, EOFError, KeyboardInterrupt):
            print("[input stream ended] ending session early")
            break
        if not user_text:
            continue
        if user_text.lower() in EXIT_COMMANDS:
            print("librarian > goodbye!")
            break
        reply = guarded_invoke(chatbot, {"input": user_text}, config=cfg)
        if reply is None:                 # provider failed mid-loop
            break
        print("librarian >", str(reply)[:400])
        transcript.append({"role": "user", "text": user_text})
        transcript.append({"role": "bot", "text": str(reply)})
    else:
        print("(turn cap %d reached)" % max_turns)
    print("=" * 64)
    return transcript

live_transcript = chat_loop("reader-live")
if live_transcript:
    print("captured %d turns from the live session" % len(live_transcript))
else:
    print("no live turns captured under automation - scripted sessions follow")

chat session 'reader-live' - cap 3 turns
you > What made Alice grow so tall?
librarian > [offline stub] Noted: "What made Alice grow so tall?" (swap llm back for real answers)
you > And how did she shrink again?
librarian > [offline stub] Noted: "And how did she shrink again?" (swap llm back for real answers)
you > quit
librarian > goodbye!
captured 4 turns from the live session


In [4]:
# --- Section 4: two scripted sessions - separation proof -----------------------
# No input() here: we drive the SAME wrapped chatbot directly with configs, so
# runs are deterministic while still exercising history injection end to end.
def scripted_chat(session_id, lines):
    """Feed fixed turns through the real wrapper; returns the reply strings."""
    cfg = {"configurable": {"session_id": session_id}}
    replies = []
    print("-" * 64)
    for line in lines:
        out = guarded_invoke(chatbot, {"input": line}, config=cfg)
        if out is None:
            break
        replies.append(str(out))
        print("[%s] you > %s" % (session_id, line))
        print("[%s] lib > %s" % (session_id, str(out)[:200]))
    return replies

scripted_chat("reader-milo", ["Hello! My name is Milo.",
                              "What is my name?",
                              "What should I read first?"])
scripted_chat("reader-june", ["Hi! My name is June.",
                              "What is my name?"])          # must say JUNE, not Milo

print("-" * 64)
print("stored sessions:")
for sid, hist in sorted(SESSION_HISTORIES.items()):
    print("   %-13s %d message(s)" % (sid, len(hist.messages)))
print("same questions, different ids, different memories - separation proven.")

----------------------------------------------------------------
[reader-milo] you > Hello! My name is Milo.
[reader-milo] lib > Welcome back to the Alice collection, Milo.
[reader-milo] you > What is my name?
[reader-milo] lib > Welcome back to the Alice collection, Milo.
[reader-milo] you > What should I read first?
[reader-milo] lib > Start at Chapter 1, 'Down the Rabbit-Hole' - everything spirals from there.
----------------------------------------------------------------
[reader-june] you > Hi! My name is June.
[reader-june] lib > Welcome back to the Alice collection, June.
[reader-june] you > What is my name?
[reader-june] lib > Welcome back to the Alice collection, June.
----------------------------------------------------------------
stored sessions:
   reader-june   4 message(s)
   reader-live   4 message(s)
   reader-milo   6 message(s)
same questions, different ids, different memories - separation proven.


In [5]:
# --- Section 5: transcript export ----------------------------------------------
# Serialize every session's stored messages to JSON so learners can diff what the
# model actually SAW versus what it replied.
ROLE_MAP = {"human": "user", "ai": "bot"}          # message.type -> readable role
TRANSCRIPTS = {}
for sid, hist in SESSION_HISTORIES.items():
    TRANSCRIPTS[sid] = [{"role": ROLE_MAP.get(m.type, m.type), "text": m.content}
                        for m in hist.messages]
if "reader-live" not in TRANSCRIPTS:               # loop ran but captured nothing
    TRANSCRIPTS["reader-live"] = []

EXPORT_PATH = DATA / "transcripts.json"
with open(EXPORT_PATH, "w", encoding="utf-8") as fh:
    json.dump(TRANSCRIPTS, fh, indent=2, ensure_ascii=True)   # ASCII-safe file too
print("exported transcripts:")
for sid, entries in TRANSCRIPTS.items():
    print("   %-13s %d entries" % (sid, len(entries)))
print("file:", EXPORT_PATH)

exported transcripts:
   reader-live   4 entries
   reader-milo   6 entries
   reader-june   4 entries
file: D:\projects\python\MLCourse\03_agentic_ai\data\transcripts.json


## Summary

- An app is four pieces: persona prompt, guarded model, parser, and the
  history wrapper - composed once, reused for every session id forever after.
- Interactive does not mean fragile: MAX_TURNS + exit commands + EOFError
  handling make the same code safe for humans AND automation.
- Scripted double-session runs are your regression tests for isolation; the
  JSON export is your audit trail of every turn stored.
- The capstone keeps exactly this skeleton and upgrades the core from
  "librarian persona" to "RAG over alice.txt with citations".